# Z5008 Big Data Lab – Assignment 2
**Student Name: [Your Name] | Roll Number: [Your Roll Number]**
**Tools used: Antigravity AI**

---

## Task 1: Environment Setup and Verification

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import pandas as pd

extra_packages = [
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262"
]

builder = SparkSession.builder \
    .appName("Z5008-Assign2-ECommerce") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "bigdata123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8")

spark = configure_spark_with_delta_pip(builder, extra_packages=extra_packages).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Spark Session Created Successfully!")

## Task 2: Data Generation and Delta Table Creation

In [ ]:
import pandas as pd
import random
from pyspark.sql.types import (
    StructType, StructField, StringType, 
    DoubleType, TimestampType
)
from pyspark.sql import functions as F

random.seed(42)
n = 5500

categories = ["Electronics", "Fashion", "Home & Garden", "Beauty", "Sports", "Groceries"]
payment_methods = ["M-Pesa", "Tigo Pesa", "Airtel Money", "Credit Card", "Cash"]
regions = ["Dar es Salaam", "Arusha", "Mwanza", "Dodoma", "Zanzibar City"]

pdf = pd.DataFrame({
    "order_id":      [f"ORD-{i+10000:05d}" for i in range(n)],
    "customer_id":   [f"CUST-{random.randint(1, 1000):04d}" for i in range(n)],
    "order_date":    pd.date_range("2026-01-01", periods=n, freq="15min"),
    "product_category": [random.choice(categories) for _ in range(n)],
    "amount_tzs":    [round(random.uniform(5000, 500000), 2) for _ in range(n)],
    "payment_method": [random.choice(payment_methods) for _ in range(n)],
    "region":        [random.choice(regions) for _ in range(n)]
})

schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), True),
    StructField("order_date", TimestampType(), True),
    StructField("product_category", StringType(), True),
    StructField("amount_tzs", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("region", StringType(), True)
])

df = spark.createDataFrame(pdf, schema=schema)
print(f"Generated {df.count()} rows in E-commerce domain.")

BRONZE_PATH = "s3a://warehouse/bronze/ecommerce_trx"

df.write.format("delta").mode("overwrite").partitionBy("product_category").save(BRONZE_PATH)
print(f"Table written to {BRONZE_PATH}")

In [ ]:
df_bronze = spark.read.format("delta").load(BRONZE_PATH)
df_bronze.createOrReplaceTempView("ecommerce_trx")

print("Query 1: Total Revenue and Average Order Value by Product Category")
spark.sql("""
    SELECT product_category, 
           COUNT(*) as order_count, 
           ROUND(SUM(amount_tzs), 2) as total_revenue, 
           ROUND(AVG(amount_tzs), 2) as avg_order_value
    FROM ecommerce_trx
    GROUP BY product_category
    ORDER BY total_revenue DESC
""").show()

print("Query 2: Payment Method Usage Analysis")
spark.sql("""
    SELECT payment_method, 
           COUNT(*) as usage_count, 
           ROUND(SUM(amount_tzs), 2) as total_processed
    FROM ecommerce_trx
    GROUP BY payment_method
    ORDER BY usage_count DESC
""").show()

## Task 3: Time Travel

In [ ]:
# Version 1: Append 500 new rows
pdf_v1 = pd.DataFrame({
    "order_id":      [f"ORD-{i+16000:05d}" for i in range(500)],
    "customer_id":   [f"CUST-{random.randint(1, 1000):04d}" for i in range(500)],
    "order_date":    pd.date_range("2026-03-01", periods=500, freq="h"),
    "product_category": [random.choice(categories) for _ in range(500)],
    "amount_tzs":    [round(random.uniform(10000, 200000), 2) for _ in range(500)],
    "payment_method": [random.choice(payment_methods) for _ in range(500)],
    "region":        [random.choice(regions) for _ in range(500)]
})
df_v1 = spark.createDataFrame(pdf_v1, schema=schema)
df_v1.write.format("delta").mode("append").save(BRONZE_PATH)
print("Version 1 created (Append 500 rows).")

# Version 2: Append 100 more rows
pdf_v2 = pd.DataFrame({
    "order_id":      [f"ORD-{i+17000:05d}" for i in range(100)],
    "customer_id":   [f"CUST-{random.randint(1, 1000):04d}" for i in range(100)],
    "order_date":    pd.date_range("2026-03-22", periods=10, freq="min").tolist() * 10,
    "product_category": [random.choice(categories) for _ in range(100)],
    "amount_tzs":    [50000.0] * 100,
    "payment_method": ["M-Pesa"] * 100,
    "region":        ["Dar es Salaam"] * 100
})
df_v2 = spark.createDataFrame(pdf_v2, schema=schema)
df_v2.write.format("delta").mode("append").save(BRONZE_PATH)
print("Version 2 created (Append 100 rows).")

In [ ]:
print("--- Table History ---")
spark.sql(f"DESCRIBE HISTORY delta.`{BRONZE_PATH}`").select("version", "timestamp", "operation", "operationMetrics").show()

print("Simulating accident and recovering...")
spark.sql(f"RESTORE TABLE delta.`{BRONZE_PATH}` TO VERSION AS OF 0")

restored_count = spark.read.format("delta").load(BRONZE_PATH).count()
print(f"Verified: Row count is back to {restored_count} (Version 0 count).")

## Task 4: MERGE / UPSERT

In [ ]:
SILVER_PATH = "s3a://warehouse/silver/ecommerce_summary"

print("Creating Silver summary table...")
(
    df_bronze.groupBy("customer_id")
    .agg(
        F.round(F.sum("amount_tzs"), 2).alias("total_spend"),
        F.count("*").alias("order_count"),
        F.max("order_date").alias("last_activity_date"),
        F.lit("active").alias("status")
    )
    .write.format("delta").mode("overwrite").save(SILVER_PATH)
)

from delta.tables import DeltaTable
silver_table = DeltaTable.forPath(spark, SILVER_PATH)

# Prepare some updates
existing_customers = spark.read.format("delta").load(SILVER_PATH).limit(20).collect()
update_ids = [r['customer_id'] for r in existing_customers[:10]]
updates_data = [(cid, 1000000.0, 100, pd.Timestamp.now(), "active") for cid in update_ids]
updates_df = spark.createDataFrame(updates_data, ["customer_id", "total_spend", "order_count", "last_activity_date", "status"])

(
    silver_table.alias("target")
    .merge(updates_df.alias("source"), "target.customer_id = source.customer_id")
    .whenMatchedUpdate(set={
        "total_spend": "source.total_spend",
        "order_count": "source.order_count",
        "last_activity_date": "source.last_activity_date"
    })
    .execute()
)
print("MERGE operation completed.")

## Task 5: Schema Evolution and Maintenance

In [ ]:
print("Schema Evolution (Adding 'customer_segment' column)")
evolution_df = df.limit(10).withColumn("customer_segment", F.lit("VIP"))

evolution_df.write.format("delta").mode("append").option("mergeSchema", "true").save(BRONZE_PATH)

print("Updated Schema:")
spark.read.format("delta").load(BRONZE_PATH).printSchema()

print("OPTIMIZE with ZORDER BY (order_id)")
spark.sql(f"OPTIMIZE delta.`{BRONZE_PATH}` ZORDER BY (order_id)")